# Evaluasi Qwen Indonesian Legal Adapter

Notebook ini menjalankan evaluator lokal untuk memeriksa integritas artefak, mengambil sampel terdeduplikasi dari `qa/test`, menjalankan inference, menghitung diagnostik tekstual, dan membuat antrean review hukum. Dataset dan model asli hanya dibaca.

Hasil otomatis tidak menyatakan model benar secara hukum. Keputusan penggunaan tetap memerlukan pemeriksaan substansi, sitasi, status peraturan, dan review manusia.

## Mode evaluasi

Default adalah `smoke`. Untuk evaluasi lebih besar, set `QWEN_LEGAL_EVAL_MODE=full` dan `QWEN_LEGAL_EVAL_LIMIT=200` sebelum menjalankan cell. Untuk gold set buatan ahli, gunakan `QWEN_LEGAL_EVAL_EXPERT_SET` dengan kolom `prompt`, `reference_answer`, dan optional `reference_source`.

Set `QWEN_LEGAL_EVAL_ADAPTER` ke path adapter lokal bila adapter belum ditempatkan pada direktori default `artifacts/local/`.

In [ ]:
from pathlib import Path
import os
import runpy

# Mendukung kernel dengan cwd di root repository maupun di folder notebooks.
cwd = Path.cwd()
script_candidates = [
    cwd / 'notebooks' / 'qwen35_legal_evaluation.py',
    cwd / 'qwen35_legal_evaluation.py',
]
script_path = next((path for path in script_candidates if path.exists()), None)
if script_path is None:
    raise FileNotFoundError(f'Evaluator tidak ditemukan; dicari di: {script_candidates}')

os.environ.setdefault('QWEN_LEGAL_EVAL_MODE', 'smoke')
# os.environ['QWEN_LEGAL_EVAL_MODE'] = 'full'
# os.environ['QWEN_LEGAL_EVAL_LIMIT'] = '200'
# os.environ['QWEN_LEGAL_EVAL_COMPARE_BASELINE'] = '1'
# os.environ['QWEN_LEGAL_EVAL_ADAPTER'] = '/path/to/final_adapter'

runpy.run_path(str(script_path), run_name='__main__')

## Cara membaca hasil

`evaluation_manifest.json` berisi provenance dan gate teknis. `model_outputs.jsonl` menyimpan prompt, referensi, jawaban model, dan diagnostik. `human_review_queue.csv` siap diberi label pada kolom `reviewer_*`.

`token_f1` dan `reference_marker_recall` hanya diagnostik overlap teks; keduanya tidak menggantikan penilaian apakah jawaban benar, lengkap, berlaku, dan didukung sumber.